# 003 - Structured Output with Prompt Engineering

This notebook shows how to get JSON output from LLMs using prompt engineering techniques for data processing.

## Setup

Import libraries and configure the API.

In [ ]:
import json
import os

import google.generativeai as genai
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get API key from environment variable
api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in environment variables. Please check your .env file.")

# Configure the API
genai.configure(api_key=api_key)
model = genai.GenerativeModel("models/gemini-2.0-flash-lite")

## Helper Function for JSON Requests

Create a function that asks for JSON output and attempts to parse it.

In [ ]:
def ask_for_json(prompt):
    """Ask the model for JSON output and try to parse it"""
    print(f"Prompt: {prompt}")
    print("-" * 60)

    response = model.generate_content(prompt)
    print(f"Response:\n{response.text}")

    print("-" * 60)

    # Return the raw response text for further processing
    return response.text

## Example 1: Basic Request

Let's start with a basic request that doesn't specify the output format.

In [ ]:
print("📝 EXAMPLE 1: BASIC REQUEST")

result1 = ask_for_json("Create a user profile with username, name, last name, and city")

## Example 2: Explicit JSON Request

Now let's explicitly request JSON format with specific field requirements.

In [ ]:
print("\n📝 EXAMPLE 2: EXPLICIT JSON REQUEST")

result2 = ask_for_json("""Create a user profile and return it as valid JSON with these exact fields:
- username (string)
- name (string)
- last_name (string)
- city (string)

Return only the JSON, no additional text.""")

## Example 3: Multiple Records

Let's request multiple user profiles as a JSON array with specific constraints.

In [ ]:
print("\n📝 EXAMPLE 3: MULTIPLE RECORDS")

result3 = ask_for_json("""Generate 3 user profiles as a JSON array. Each user must have:
- username: unique, lowercase with underscores
- name: realistic first name
- last_name: realistic last name
- city: real city name

Return only the JSON array, no markdown formatting.""")

## Create DataFrame (Optional)

If you want to process the JSON data into a pandas DataFrame, you can manually parse the responses and create a DataFrame.

## JSON Parsing Function

Now let's create a function to parse the JSON responses from the LLM outputs.

In [ ]:
def parse_json_response(response_text):
    """Parse JSON from LLM response text"""
    if not response_text:
        print("❌ No response text to parse")
        return None

    # Clean up the response text
    response_text = response_text.strip()

    # Remove markdown code blocks if present
    if response_text.startswith('```json'):
        response_text = response_text[7:]
    elif response_text.startswith('```'):
        response_text = response_text[3:]

    if response_text.endswith('```'):
        response_text = response_text[:-3]

    # Try to parse as JSON
    try:
        parsed_json = json.loads(response_text)
        print("✅ Successfully parsed JSON:")
        print(json.dumps(parsed_json, indent=2))
        return parsed_json
    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing failed: {e}")
        print(f"Raw text: {response_text[:200]}...")
        return None

## Parse the Results

Now let's parse the JSON responses from our previous examples.

In [ ]:
# Parse the results from our previous examples
print("Parsing results from the examples above...\n")

# Parse result1 (basic request)
print("🔍 Parsing Result 1:")
parsed_result1 = parse_json_response(result1)

# Parse result2 (explicit JSON request)
print("\n🔍 Parsing Result 2:")
parsed_result2 = parse_json_response(result2)

# Parse result3 (multiple records)
print("\n🔍 Parsing Result 3:")
parsed_result3 = parse_json_response(result3)

In [ ]:
# Create DataFrame from successfully parsed results
all_users = []

# Collect data from parsed results
for i, result in enumerate([parsed_result1, parsed_result2, parsed_result3], 1):
    print(f"Processing result {i}...")
    if result:
        if isinstance(result, list):
            all_users.extend(result)
            print(f"  ✅ Added {len(result)} users from list")
        elif isinstance(result, dict):
            all_users.append(result)
            print("  ✅ Added 1 user from dict")
    else:
        print(f"  ❌ No valid data in result {i}")

# Create DataFrame if we have any valid data
if all_users:
    df = pd.DataFrame(all_users)
    print(f"\n📊 CREATED DATAFRAME ({len(df)} users):")
    print(df.to_string(index=False))

    # Save to CSV
    df.to_csv('user_profiles_simple.csv', index=False)
    print("\n💾 Saved to 'user_profiles_simple.csv'")
else:
    print("\n❌ No valid user data found to create DataFrame")
    print("This could happen if the LLM responses weren't in valid JSON format")

## Key Observations

- **Basic requests** often return unstructured text
- **Explicit JSON requests** improve the chances of getting valid JSON
- **Detailed specifications** help ensure consistent field names and types
- **Prompt engineering** is not always reliable for structured output
- Manual parsing and error handling is often required